In [1]:
import pandas as pd
import sys
sys.path.append('..')
from src.config import CLEAN_DATA_PATH

df = pd.read_csv(CLEAN_DATA_PATH, parse_dates=['InvoiceDate'])
trading_days = df['InvoiceDate'].dt.normalize().drop_duplicates().sort_values().reset_index(drop=True)
gaps = trading_days.diff().dt.days

closures = pd.DataFrame({
    'last_trading_day': trading_days.shift(1)[gaps > 3],
    'next_trading_day': trading_days[gaps > 3],
    'gap_days': gaps[gaps > 3]
})
print(closures)

    last_trading_day next_trading_day  gap_days
21        2009-12-23       2010-01-04      12.0
97        2010-04-01       2010-04-06       5.0
319       2010-12-23       2011-01-04      12.0
412       2011-04-21       2011-04-26       5.0


In [2]:
import pandas as pd
import sys
sys.path.append('..')
from src.config import NON_PRODUCT_CODES

raw = pd.concat(
    pd.read_excel('../data/online_retail_II.xlsx', sheet_name=None).values(),
    ignore_index=True
)
raw['Invoice'] = raw['Invoice'].astype(str)
raw['Is_Cancel'] = raw['Invoice'].str.startswith('C')

cols = ['Invoice', 'StockCode', 'Description', 'Quantity', 'Price', 'InvoiceDate']

# 1. Customer 12346: is the 74,215-unit order followed by a cancellation?
print("Customer 12346, 2011 transactions:")
print(raw[(raw['Customer ID'] == 12346) & (raw['InvoiceDate'] >= '2011-01-01')][cols])

# 2. The 80,995-unit order: who placed it, and was it cancelled?
big = raw[raw['Quantity'].abs() == 80995][cols + ['Customer ID']]
print("\nAll lines with |Quantity| = 80,995:")
print(big)

# 3. Overall scale of cancellations (product lines with a Customer ID only)
product = ~raw['StockCode'].astype(str).isin(NON_PRODUCT_CODES) & raw['Customer ID'].notna()
cancels = raw[product & raw['Is_Cancel']]
sales = raw[product & ~raw['Is_Cancel'] & (raw['Quantity'] > 0) & (raw['Price'] > 0)]

cancel_value = (-cancels['Quantity'] * cancels['Price']).sum()
sales_value = (sales['Quantity'] * sales['Price']).sum()
print(f"\nCancellation lines: {len(cancels):,}")
print(f"Cancelled value: £{cancel_value:,.0f}")
print(f"Gross sales value: £{sales_value:,.0f}")
print(f"Cancelled as % of gross sales: {cancel_value / sales_value * 100:.1f}%")

Customer 12346, 2011 transactions:
        Invoice StockCode                     Description  Quantity  Price  \
587080   541431     23166  MEDIUM CERAMIC TOP STORAGE JAR     74215   1.04   
587085  C541433     23166  MEDIUM CERAMIC TOP STORAGE JAR    -74215   1.04   

               InvoiceDate  
587080 2011-01-18 10:01:00  
587085 2011-01-18 10:17:00  

All lines with |Quantity| = 80,995:
         Invoice StockCode                  Description  Quantity  Price  \
1065882   581483     23843  PAPER CRAFT , LITTLE BIRDIE     80995   2.08   
1065883  C581484     23843  PAPER CRAFT , LITTLE BIRDIE    -80995   2.08   

                InvoiceDate  Customer ID  
1065882 2011-12-09 09:15:00      16446.0  
1065883 2011-12-09 09:27:00      16446.0  

Cancellation lines: 17,983
Cancelled value: £729,146
Gross sales value: £17,438,960
Cancelled as % of gross sales: 4.2%


In [3]:
df = raw.drop_duplicates()
product = ~df['StockCode'].astype(str).isin(NON_PRODUCT_CODES) & df['Customer ID'].notna()

cancels = df[product & df['Is_Cancel']].copy()
sales = df[product & ~df['Is_Cancel'] & (df['Quantity'] > 0) & (df['Price'] > 0)].copy()

cancels['Abs_Qty'] = -cancels['Quantity']
cancels['Value'] = cancels['Abs_Qty'] * cancels['Price']
cancels = cancels.reset_index().rename(columns={'index': 'cancel_idx'})
sales = sales.reset_index().rename(columns={'index': 'sale_idx'})

keys = ['Customer ID', 'StockCode']

# Case 1: exact pairs (same customer, product, quantity, price; purchase at or before cancellation)
exact = cancels.merge(
    sales[keys + ['Quantity', 'Price', 'InvoiceDate', 'sale_idx']],
    left_on=keys + ['Abs_Qty', 'Price'],
    right_on=keys + ['Quantity', 'Price'],
    suffixes=('', '_sale')
)
exact = exact[exact['InvoiceDate_sale'] <= exact['InvoiceDate']]
exact_ids = set(exact['cancel_idx'])

# Case 2 vs 3: among the rest, does any earlier purchase of that product by that customer exist?
rest = cancels[~cancels['cancel_idx'].isin(exact_ids)]
prior = rest.merge(sales[keys + ['InvoiceDate']], on=keys, suffixes=('', '_sale'))
prior = prior[prior['InvoiceDate_sale'] <= prior['InvoiceDate']]
partial_ids = set(prior['cancel_idx'])

groups = {
    'Case 1: exact pair': cancels[cancels['cancel_idx'].isin(exact_ids)],
    'Case 2: partial / non-exact': rest[rest['cancel_idx'].isin(partial_ids)],
    'Case 3: unmatched': rest[~rest['cancel_idx'].isin(partial_ids)],
}

total = cancels['Value'].sum()
print(f"Total cancelled value (deduplicated): £{total:,.0f}\n")
for name, part in groups.items():
    v = part['Value'].sum()
    print(f"{name:30s} lines: {len(part):6,}   value: £{v:>10,.0f}   ({v / total * 100:5.1f}%)")

print("\nTop 10 cancellations by value:")
print(cancels.nlargest(10, 'Value')[['Invoice', 'Customer ID', 'Description', 'Abs_Qty', 'Price', 'Value', 'InvoiceDate']])

# Where does the fake £168k currently sit in the modeling data?
from src.config import MODELING_DATA_PATH
mt = pd.read_csv('..' + '/' + MODELING_DATA_PATH if not MODELING_DATA_PATH.startswith('..') else MODELING_DATA_PATH)
if 'Customer ID' in mt.columns:
    mt = mt.set_index('Customer ID')
print("\nTop 5 CLV_Target values (6-month table):")
print(mt['CLV_Target'].nlargest(5))

Total cancelled value (deduplicated): £719,446

Case 1: exact pair             lines:  6,107   value: £   548,070   ( 76.2%)
Case 2: partial / non-exact    lines:  9,953   value: £   132,312   ( 18.4%)
Case 3: unmatched              lines:  1,576   value: £    39,064   (  5.4%)

Top 10 cancellations by value:
       Invoice  Customer ID                         Description  Abs_Qty  \
17630  C581484      16446.0         PAPER CRAFT , LITTLE BIRDIE    80995   
10073  C541433      12346.0      MEDIUM CERAMIC TOP STORAGE JAR    74215   
11811  C550456      15749.0  FAIRY CAKE FLANNEL ASSORTED COLOUR     3114   
11809  C550456      15749.0  WHITE HANGING HEART T-LIGHT HOLDER     1930   
11807  C550456      15749.0                  DOORMAT FAIRY CAKE      670   
14807  C567527      17450.0               PANTRY CHOPPING BOARD      756   
11810  C550456      15749.0         GIN + TONIC DIET METAL SIGN     2000   
1818   C498151      13902.0     SMALL FAIRY CAKE FRIDGE MAGNETS     2504   
11808

In [4]:
import importlib
import src.cleaning
importlib.reload(src.cleaning)
from src.cleaning import net_cancellations

# --- Synthetic test: known answer ---
t = pd.Timestamp
test = pd.DataFrame({
    'Invoice':     ['1', '2', 'C3', '4', 'C5', 'C6'],
    'Customer ID': [1, 1, 1, 1, 1, 2],
    'StockCode':   ['A', 'A', 'A', 'A', 'A', 'A'],
    'Quantity':    [10, 5, -8, 7, -7, -3],
    'Price':       [1.0, 1.0, 1.0, 1.0, 1.0, 1.0],
    'InvoiceDate': [t('2010-01-01'), t('2010-01-02'), t('2010-01-03'),
                    t('2010-01-04'), t('2010-01-04 01:00'), t('2010-01-05')]
})
netted, s = net_cancellations(test)
print(netted[['Invoice', 'Quantity']])
print(s)

  Invoice  Quantity
0       1         7
{'cancellation_lines': 3, 'fully_netted': 2, 'partly_netted': 0, 'unmatched': 1, 'value_netted': 15.0, 'value_unmatched_discarded': 3.0, 'purchase_rows_reduced': 3, 'purchase_rows_removed_entirely': 2}


In [5]:
df = raw.drop_duplicates()
df = df[df['Customer ID'].notna()]
df = df[~df['StockCode'].astype(str).isin(NON_PRODUCT_CODES)]

netted, summary = net_cancellations(df)
for k, v in summary.items():
    print(f"{k:32s} {v:,}")

# Both mega-orders must be gone
print("\n12346 jar order still present:", ((netted['Customer ID'] == 12346) & (netted['StockCode'] == 23166)).any())
print("16446 Little Birdie order still present:", ((netted['Customer ID'] == 16446) & (netted['StockCode'] == 23843)).any())

cancellation_lines               17,636
fully_netted                     15,880
partly_netted                    126
unmatched                        1,630
value_netted                     730,405.01
value_unmatched_discarded        43,573.44
purchase_rows_reduced            15,716
purchase_rows_removed_entirely   6,031

12346 jar order still present: False
16446 Little Birdie order still present: False


In [6]:
importlib.reload(src.cleaning)
from src.cleaning import net_cancellations

netted, summary, log = net_cancellations(df, return_log=True)

log['Value_Sale'] = log['Qty'] * log['Sale_Price']
log['Value_Cancel'] = log['Qty'] * log['Cancel_Price']
log['Gap'] = log['Value_Sale'] - log['Value_Cancel']

print(f"Netted at sale prices:   £{log['Value_Sale'].sum():,.0f}")
print(f"Netted at cancel prices: £{log['Value_Cancel'].sum():,.0f}")
print(f"Gap:                     £{log['Gap'].sum():,.0f}")

diff = log[log['Sale_Price'] != log['Cancel_Price']]
print(f"\nAllocations at a different price: {len(diff):,} of {len(log):,}")
print(f"  of which sale price higher: {(diff['Gap'] > 0).sum():,}   lower: {(diff['Gap'] < 0).sum():,}")

# Hypothesis test: when prices differed, did a same-price purchase exist that LIFO skipped?
sales_all = df[~df['Invoice'].astype(str).str.startswith('C') & (df['Quantity'] > 0)]
same_price_available = diff.merge(
    sales_all[['Customer ID', 'StockCode', 'Price', 'InvoiceDate']],
    left_on=['Customer ID', 'StockCode', 'Cancel_Price'],
    right_on=['Customer ID', 'StockCode', 'Price']
)
same_price_available = same_price_available[same_price_available['InvoiceDate'] <= same_price_available['Cancel_Date']]
n_skipped = same_price_available['Cancel_Invoice'].nunique()
print(f"\nMismatched allocations where a same-price earlier purchase existed: "
      f"{n_skipped:,} cancellation invoices")

print("\nTop 10 allocations by gap:")
print(log.nlargest(10, 'Gap')[['Cancel_Invoice', 'Sale_Invoice', 'Customer ID', 'StockCode',
                               'Qty', 'Cancel_Price', 'Sale_Price', 'Gap', 'Sale_Date', 'Cancel_Date']])

Netted at sale prices:   £730,405
Netted at cancel prices: £675,873
Gap:                     £54,532

Allocations at a different price: 1,035 of 16,356
  of which sale price higher: 397   lower: 638

Mismatched allocations where a same-price earlier purchase existed: 233 cancellation invoices

Top 10 allocations by gap:
      Cancel_Invoice Sale_Invoice  Customer ID StockCode   Qty  Cancel_Price  \
11468        C556448       556444      15098.0     22502    59          4.95   
7864         C536757       530715      15838.0     84347  9360          0.03   
11467        C556448       556446      15098.0     22502     1          4.95   
5953         C525699       524874      12931.0     20829   200          0.41   
2098         C504841       503955      13694.0     84919   160          4.00   
7576         C534845       518674      14646.0     22656     2        150.00   
4356         C516883       515602      14895.0     22656     1         88.50   
2097         C504841       503955     

In [7]:
import pandas as pd
import sys
sys.path.append('../src')
from config import CLEAN_DATA_PATH

clean = pd.read_csv(CLEAN_DATA_PATH, parse_dates=['InvoiceDate'])
codes = clean['StockCode'].astype(str)

# Standard product codes: 5 digits, optionally followed by 1-2 letters
is_standard = codes.str.fullmatch(r'\d{5}[A-Za-z]{0,2}')

odd = clean[~is_standard].copy()
odd['Code'] = codes[~is_standard]
odd['Revenue'] = odd['Quantity'] * odd['Price']

audit = odd.groupby('Code').agg(
    Rows=('Invoice', 'size'),
    Customers=('Customer ID', 'nunique'),
    Revenue=('Revenue', 'sum'),
    Median_Price=('Price', 'median'),
    Example_Description=('Description', 'first')
).sort_values('Revenue', ascending=False)

print(f"Non-standard codes: {len(audit)}   rows: {len(odd):,}   revenue: £{odd['Revenue'].sum():,.0f}")
print(f"Share of all clean rows: {len(odd) / len(clean) * 100:.2f}%")
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 45)
audit

Non-standard codes: 6   rows: 63   revenue: £4,497
Share of all clean rows: 0.01%


,Rows,Customers,Revenue,Median_Price,Example_Description
Code,,,,,
ADJUST,32,25,3538.520,79.565,Adjustment by john on 26/01/2010 16
ADJUST2,3,3,731.050,300.130,Adjustment by Peter on Jun 25 2010
TEST001,9,2,211.500,4.500,This is a test product.
SP1002,2,2,14.750,2.950,KID'S CHALKBOARD/EASEL
TEST002,1,1,1.000,1.000,This is a test product.
PADS,16,14,0.016,0.001,PADS TO MATCH ALL CUSHIONS
